# Cinemática Inversa Numérica
---

In [90]:
import numpy as np
# from sympy import sym
from numpy import float32, float64, int32, int64

from dataclasses import dataclass
from typing import Self

Utilizamos alguna semilla fila, para que las pruebas a pesar de ser aleatoria, sean determinista para cada corrida del código y utilizar los resultados estadísticos

In [91]:
np.random.seed(0)

In [92]:
CANT_EJES = 6

## Definición del Robot
---
En esta sección vamos a definir la Pose, el Robot IRB140 con su configuración y dimensiones, para usarlos a lo largo del trabajo. Se define la cinemática directa del mismo ya que será necesario para calcula la cinemática inversa de forma numérica

In [139]:
def problema_inverso_rotacion(rotacion: np.ndarray, epsilon: float64 = 1e-9) -> np.ndarray:
  theta = np.arccos((np.trace(rotacion) - 1) / 2)

  sen_theta = np.sin(theta)
  if abs(sen_theta) > epsilon:
    kx = (rotacion[2, 1] - rotacion[1, 2]) / (2 * sen_theta)
    ky = (rotacion[0, 2] - rotacion[2, 0]) / (2 * sen_theta)
    kz = (rotacion[1, 0] - rotacion[0, 1]) / (2 * sen_theta)

  else:
    kx = np.sqrt((rotacion[0, 0] + 1) / 2)
    ky = np.sqrt((rotacion[1, 1] + 1) / 2)
    kz = np.sqrt((rotacion[2, 2] + 1) / 2)

  k = np.array([kx, ky, kz])
  return theta * k / np.linalg.norm(k)

def matriz_antisimetrica(vector: np.ndarray) -> np.ndarray:
  x, y, z = tuple(vector.tolist())
  return np.array([
      [  0, -z,  y ],
      [  z,  0, -x ],
      [ -y,  x,  0 ],
  ])

In [93]:
@dataclass
class Pose:
  rotacion: np.ndarray
  translacion: np.ndarray

  @classmethod
  def identidad(cls) -> Self:
    return Pose(np.eye(3), np.zeros(3))

  def __mul__(self, pose: Self) -> Self:
    return Pose(
      self.rotacion @ pose.rotacion,
      self.rotacion @ pose.translacion + self.translacion,
    )

  def __sub__(self, pose: Self) -> np.ndarray:
    diff_rotacion = problema_inverso_rotacion(self.rotacion.T @ pose.rotacion)
    diff_translacion = self.translacion - pose.translacion
    return np.hstack(( diff_translacion, diff_rotacion ))

  def inv(self) -> Self:
    return Pose(
        self.rotacion.T,
        -self.rotacion.T @ self.translacion,
    )

  def distancia_cuadrado(self, pose: Self) -> float64:
    diff = self - pose
    return diff[0]**2 + diff[1]**2 + diff[2]**2

In [94]:
def DH(theta: float64, d: float64, a: float64, alpha: float64) -> Pose:
    cth = np.cos(theta)
    sth = np.sin(theta)
    ca = np.cos(alpha)
    sa = np.sin(alpha)

    return Pose(np.array([
        [cth, -sth*ca,  sth*sa],
        [sth,  cth*ca, -cth*sa],
        [  0,     sa,       ca],
    ]), np.array([ a*cth, a*sth, d ]))

In [140]:
@dataclass
class Dimensiones:
  d1: float64
  a1: float64
  a2: float64
  d4: float64
  d6: float64

@dataclass
class Configuracion:
  c1: bool # True es lo que vimos como 1, y Flase como -1
  c2: bool
  c3: bool

  def __eq__(self, otro: Self) -> bool:
    return self.c1 == otro.c1 and self.c2 == otro.c2 and self.c3 == otro.c3

class IRB140:
  def __init__(self, dim: Dimensiones, ABase0: Pose | None = None, A6Tool: Pose | None = None):
    self.dim = dim

    self.A01 = lambda q1: DH(q1, dim.d1, dim.a1, -np.pi/2)
    self.A12 = lambda q2: DH(q2,      0, dim.a2,        0)
    self.A23 = lambda q3: DH(q3,      0,      0,  np.pi/2)
    self.A34 = lambda q4: DH(q4, dim.d4,      0, -np.pi/2)
    self.A45 = lambda q5: DH(q5,      0,      0,  np.pi/2)
    self.A56 = lambda q6: DH(q6, dim.d6,      0,        0)

    self.ABase0 = Pose.identidad() if ABase0 is None else ABase0
    self.A6Tool = Pose.identidad() if A6Tool is None else A6Tool

  def cinematica_directa(self, q: np.ndarray) -> Pose:
    q1, q2, q3, q4, q5, q6 = tuple(q.tolist())
    A06 = self.A01(q1) * self.A12(q2) * self.A23(q3) * self.A34(q4) * self.A45(q5) * self.A56(q6)
    return self.ABase0 * A06 * self.A6Tool

  def cinematica_velocidad(self, q: np.ndarray) -> np.ndarray:
    q1, q2, q3, q4, q5, q6 = tuple(q.tolist())

    As = [self.A01(q1), self.A12(q2), self.A23(q3), self.A34(q4), self.A45(q5), self.A56(q6)]
    Aacc = Pose.identidad()

    Jn = np.zeros((CANT_EJES, CANT_EJES))
    for i in range(CANT_EJES - 1, -1, -1):
      # Aacc = I
      # necesito generar A^n_{n-1}, A^n_{n-2}, ..., A^n_0
      # Aacc = A^n_{n-2} = A^{n-1}_{n-2} @ A^n_{n-1}
      Aacc = As[i] * Aacc
      p = Aacc.translacion
      x = Aacc.rotacion[:, 0]
      y = Aacc.rotacion[:, 1]
      z = Aacc.rotacion[:, 2]

      for j, versor in enumerate((x, y, z)):
        Jn[j, i] = np.cross(p, versor)[2]
        Jn[j + 3, i] = versor[2]

    RBase6 = (self.ABase0 * Aacc).rotacion
    p6Tool = self.A6Tool.translacion
    JnBase = np.vstack((
        np.hstack(( RBase6, -RBase6 @ matriz_antisimetrica(p6Tool) )),
        np.hstack(( np.zeros(RBase6.shape), RBase6 ))
    ))

    return JnBase @ Jn

  def configuracion(self, q: np.ndarray, epsilon: float64 = 1e-9) -> Configuracion | None:
    _, q2, q3, _, q5, _ = tuple(q.tolist())

    config1 = self.dim.d4 * np.sin(q2 + q3) + self.dim.a2 * np.cos(q2) + self.dim.a1
    config2 = np.cos(q3)
    config3 = q5 + 0.0001

    if abs(config1) < epsilon or abs(config2) < epsilon or abs(config3) < epsilon:
      return None

    return Configuracion(bool(config1 > 0), bool(config2 > 0), bool(config3 > 0))

irb140 = IRB140(Dimensiones(0.352, 0.070, 0.360, 0.380, 0))

In [141]:
q_seed = np.array([0.2, -0.5 - np.pi/2, 0.3 + np.pi, 1.0, -0.7, 0.2])
mejor = irb140.cinematica_velocidad(q_seed)

np.set_printoptions(precision=3, suppress=True)
print(mejor)

[[-0.054  0.384  0.074  0.     0.     0.   ]
 [ 0.264  0.078  0.015  0.     0.     0.   ]
 [-0.    -0.2   -0.372  0.     0.     0.   ]
 [-0.    -0.199 -0.199  0.961 -0.271  0.775]
 [-0.     0.98   0.98   0.195  0.496 -0.396]
 [ 1.    -0.    -0.     0.199  0.825  0.493]]


## Cinemática inversa naïve
---
La forma que plantearemos para resolver la cinemática inversa es a partir de generar una función de costo $J(\bar{q})$, la cual el algoritmo va a intentar minimizar

Por lo tanto, para resolver la minimización utilizaremos el algoritmo del gradiente descendiente, el cual es un método iterativo que dado una semilla $\bar{q}_0$, la va actualizando en dirección opuesta al gradiente $\nabla J(\bar{q})$ de la siguiente manera $$ \bar{q}_{n + 1} = \bar{q}_{n} - \alpha \nabla J(\bar{q}) $$ donde $\alpha$ es un hiperparámetro del algoritmo

Finalmente, para lograr que el mínimo que encuentre el algoritmo sea la Pose destino $\text{Pose}_{dest}$ que estamos buscando, plantearemos la función de costo como $$ J(\bar{q}) = \left\lVert \text{Pose}_{base}^{tool}(\bar{q}) - \text{Pose}_{dest} \right\rVert^2 $$ donde $\text{Pose}_{base}^{tool}(\bar{q})$ representa la cinemática directa   

In [98]:
def cinematima_inversa(
  pose_destino: Pose,
  alfa: float64,
  epsilon: float64 = 1e-2,
  max_iteraciones: int32 = 1000,
  variacion: float64 = 1e-4,
) -> tuple[np.ndarray, bool]:
  # Semilla inicial al algoritmo
  q = np.array([0., 0., 0., 0., np.pi / 2, 0.])

  termina_correctamente = False
  for _ in range(max_iteraciones):
    costo = funcion_costo(irb140.cinematica_directa(q), pose_destino)
    if costo < epsilon:
      termina_correctamente = True
      break

    gradiente = np.zeros(CANT_EJES)
    for i in range(CANT_EJES):
      delta_q = np.zeros(6)
      delta_q[i] += variacion

      delta_costo = funcion_costo(irb140.cinematica_directa(q + delta_q), pose_destino) - costo

      gradiente[i] = delta_costo / variacion
    print(f"Gradiente: {gradiente}")

    q -= alfa * gradiente
    print("Q:", q)
    print("Pose:", irb140.cinematica_directa(q))
    print("-" * 120)

  return np.mod(q + np.pi, 2 * np.pi) - np.pi, termina_correctamente

Para confirmar el funcionamiento, realizaremos pruebas generando $100$ valores de $\bar{q}$ aleatorios, los cuales nos aseguramos que tenga su correspondiente inversa asegurandonos que tenga una configuración valida, y por cinemática directa calcular su correspondiente pose, la cual será la pose destino utilizada en el calculo de la cinemática inversa

In [99]:
def generar_angulos() -> tuple[np.ndarray, Configuracion]:
  q = np.zeros(CANT_EJES)
  config = None

  while config is None:
    # definimos los q ∈ [-π, π]
    q = 2 * np.pi * (np.random.rand(CANT_EJES) - 0.5)
    config = irb140.configuracion(q)

  return q, config


In [100]:
CANT_PRUEBAS = 1
print(f"Numero de pruebas: {CANT_PRUEBAS}")

cant_acertada = 0
for _ in range(CANT_PRUEBAS):
  q_destino, _ = generar_angulos()
  print("destino: ", q_destino)
  print("Pose:", irb140.cinematica_directa(q_destino))
  print("-" * 120)
  q_obtenido, _ = cinematima_inversa(irb140.cinematica_directa(q_destino), 1e-6, max_iteraciones = 10)

  if np.linalg.norm(q_destino - q_obtenido) < 1e-1:
    cant_acertada += 1

print(f"\tResultado: {cant_acertada}/{CANT_PRUEBAS}")

Numero de pruebas: 1
destino:  [-0.39215112  2.46158236  2.91327904 -0.73235854  1.83296247  0.18155214]
Pose: Pose(rotacion=array([[ 0.76948759,  0.53406004,  0.35024095],
       [ 0.01149547,  0.53672752, -0.84367732],
       [-0.6385583 ,  0.65322542,  0.40686589]]), translacion=array([-470.86399937,  194.73603769,    7.35753647]))
------------------------------------------------------------------------------------------------------------------------
Gradiente: [-167493.23928845  416334.87357059  684656.9179662        0.
       0.               0.        ]
Q: [ 0.16749324 -0.41633487 -0.68465692  0.          1.57079633  0.        ]
Pose: Pose(rotacion=array([[ 8.79178744e-01, -1.66711195e-01,  4.46376650e-01],
       [ 1.48649170e-01,  9.86005769e-01,  7.54721596e-02],
       [-4.52712007e-01,  6.63410799e-18,  8.91656794e-01]]), translacion=array([ 59.57273844,  10.07239787, 317.61857124]))
--------------------------------------------------------------------------------------------

Se obtiene el resultado de $10$ entre las $100$ pruebas que se intentaron, esto muestra que a pesar de obtener resultados correctos, el algoritmo necesita más información para obtener el $100\%$ de aciertos

Esto ocurre ya que el problema que resuelve el gradiente descendiente es de minimización. Como conocemos el robot, sabemos que para una dada pose pueden existir multiples configuraciones del robot que lleven el end-effector a la pose destino, y esta implementación de la cinemática inversa no puede distinguir entre estas configuraciones dandonos como resultado los $\bar{q}$ más cercanos a la configuración de la semilla inicial

## Cinemática inversa con más información
---
Como vimos se tiene que incorporar más información al algoritmo para obtener el $100\%$ de aciertos. Esta información puede ser dada por la configuración de la pose destino, por lo que probarémos utilizar una semilla que tenga la misma configuración

In [101]:
def semilla(config: Configuracion) -> np.ndarray:
  q2 = 0
  if not config.c1:
    alpha = np.atan2(irb140.dim.d4, irb140.dim.a2)
    q2 = alpha - np.pi if config.c2 else np.pi - alpha

  q3 = 0 if config.c2 else np.pi
  q5 = np.pi / 2 if config.c3 else -np.pi / 2

  if irb140.dim.d4 * np.sin(q2 + q3) + irb140.dim.a2 * np.cos(q2) + irb140.dim.a1 > 0 and not config.c1:
    raise ValueError("no se puede obtener esta configuración con las dimensiones del robot")

  match (config.c1, config.c2, config.c3):
    case (( True,  True,  True)): return np.array([ 0., q2, q3, 0., q5, 0., ])
    case (( True,  True, False)): return np.array([ 0., q2, q3, 0., q5, 0., ])
    case (( True, False,  True)): return np.array([ 0., q2, q3, 0., q5, 0., ])
    case (( True, False, False)): return np.array([ 0., q2, q3, 0., q5, 0., ])
    case ((False,  True,  True)): return np.array([ 0., q2, q3, 0., q5, 0., ])
    case ((False,  True, False)): return np.array([ 0., q2, q3, 0., q5, 0., ])
    case ((False, False,  True)): return np.array([ 0., q2, q3, 0., q5, 0., ])
    case ((False, False, False)): return np.array([ 0., q2, q3, 0., q5, 0., ])

In [102]:
def cinematima_inversa(
  pose_destino: Pose,
  config: Configuracion,
  alfa: float64,
  epsilon: float64 = 1e-2,
  max_iteraciones: int32 = 1000,
  variacion: float64 = 1e-4,
) -> tuple[np.ndarray, bool]:
  # Semilla inicial al algoritmo determinada por la configuración
  q = semilla(config)

  termina_correctamente = False
  for k in range(max_iteraciones):
    costo = funcion_costo(irb140.cinematica_directa(q), pose_destino)
    if costo < epsilon:
      termina_correctamente = True
      break

    gradiente = np.zeros(6)
    for i in range(6):
      delta_q = np.zeros(6)
      delta_q[i] += variacion

      delta_costo = funcion_costo(irb140.cinematica_directa(q + delta_q), pose_destino) - costo

      gradiente[i] = delta_costo / variacion

    nuevo_q = q - alfa * gradiente
    nuevo_q = np.mod(nuevo_q + np.pi, 2 * np.pi) - np.pi

    nueva_config = irb140.configuracion(nuevo_q)
    if nueva_config is not None and config == nueva_config:
      q = nuevo_q
    else:
      alfa /= 10
      # print(f"Iter: {k}, Reduciendo alfa a {alfa}")

  return q, termina_correctamente

In [103]:
CANT_PRUEBAS = 100
print(f"Numero de pruebas con configuración: {CANT_PRUEBAS}")

cant_acertada = 0
for _ in range(CANT_PRUEBAS):
  q_destino, config = generar_angulos()
  q_obtenido, _ = cinematima_inversa(irb140.cinematica_directa(q_destino), config, 1e-6, max_iteraciones = 100)

  if np.linalg.norm(q_destino - q_obtenido) < 1e-1:
    cant_acertada += 1
  else:
    print("-" * 100)
    print(f"Esperado: {q_destino} y {config}\nObtenido: {q_obtenido} y {irb140.configuracion(q_obtenido)}")
    print(f"Pose esperada: {irb140.cinematica_directa(q_destino)}\nPose obtenida: {irb140.cinematica_directa(q_obtenido)}")

print(f"\tResultado: {cant_acertada}/{CANT_PRUEBAS}")

Numero de pruebas con configuración: 100
----------------------------------------------------------------------------------------------------
Esperado: [ 0.42753659  2.67410254 -2.69525994 -2.59414312 -3.01455672  2.08991213] y Configuracion(c1=False, c2=False, c3=False)
Obtenido: [ 0.42720326  2.67411796 -2.69534236  0.         -1.57079633  0.        ] y Configuracion(c1=False, c2=False, c3=False)
Pose esperada: Pose(rotacion=array([[ 0.44369099, -0.89163175,  0.09017279],
       [-0.89402087, -0.43339573,  0.11355543],
       [-0.06216913, -0.13099987, -0.98943117]]), translacion=array([-236.06229653, -107.56009306,  217.68198853]))
Pose obtenida: Pose(rotacion=array([[-1.93154756e-02, -4.14327032e-01, -9.09923086e-01],
       [-8.79318407e-03,  9.10128074e-01, -4.14233714e-01],
       [ 9.99774771e-01,  6.25318623e-17, -2.12228105e-02]]), translacion=array([-236.12358275, -107.4929849 ,  217.68640203]))
--------------------------------------------------------------------------------